# Worksheet Part B: tuning the pSTAT pulse

Companion to `Worksheets/module1-exercises.pdf`. With SOCS feedback, total pSTAT rises to a peak and then
falls to a lower adapted level. `pulse()` measures three properties of that response:

- **time of peak** (min)
- **peak height** (µM)
- **adapted level** (µM): the late steady state. The response oscillates slightly before it settles, so we simulate to t = 600 min.

**Fill in the prediction table (B1) on the worksheet before you run the cells below.**

In [ ]:
import bngsim
import numpy as np
import matplotlib.pyplot as plt

model = bngsim.Model.from_bngl("../models/stat_step6_feedback.bngl")
sim = bngsim.Simulator(model, method="ode")

def pulse(param=None, factor=1.0):
    """Time of peak, peak height, and adapted level of total pSTAT."""
    if param is not None:
        default = model.get_param(param)
        model.set_param(param, factor * default)
    model.reset()
    res = sim.run(t_span=(0, 600), n_points=6001)
    if param is not None:
        model.set_param(param, default)        # restore
    y = res.observables["pSTAT"]
    i = np.argmax(y)
    return res.time[i], y[i], y[-1]

## Default parameters

In [ ]:
t_peak, height, adapted = pulse()
print(f"time of peak = {t_peak:.1f} min, peak height = {height:.2f} uM, adapted level = {adapted:.3f} uM")

## B2. Test your predictions

Double each parameter and compare with the default. Changes smaller than 5% are shown as `—`.

In [ ]:
def arrow(new, old, tol=0.05):
    r = new / old - 1
    return "—" if abs(r) < tol else ("↑" if r > 0 else "↓")

base = pulse()
print(f"{'doubled':10s} {'time of peak':>16s} {'peak height':>16s} {'adapted level':>16s}")
print(f"{'none':10s} {base[0]:14.1f}   {base[1]:14.2f}   {base[2]:14.3f}")
for p in ["L0", "kact", "kdiss", "kdephos", "ktl", "kSOCSdeg"]:
    r = pulse(p, 2)
    print(f"{p:10s} {r[0]:14.1f} {arrow(r[0], base[0])} {r[1]:14.2f} {arrow(r[1], base[1])} "
          f"{r[2]:14.3f} {arrow(r[2], base[2])}")

## Look at the curves

Change `param` to see how the whole response changes, not just the three numbers.

In [ ]:
param = "kSOCSdeg"
default = model.get_param(param)
for f in [0.5, 1, 2]:
    model.set_param(param, f * default)
    model.reset()
    res = sim.run(t_span=(0, 600), n_points=6001)
    plt.plot(res.time, res.observables["pSTAT"], label=f"{param} = {f} x default")
model.set_param(param, default)
plt.xlim(0, 200)
plt.xlabel("time (min)")
plt.ylabel("total pSTAT (uM)")
_ = plt.legend()

## B3, question 2: does ligand matter at low concentration?

At the default `L0 = 10` uM almost every receptor is bound. Repeat the ligand test at `L0 = 0.1` uM.

In [ ]:
model.set_param("L0", 0.1)
for label, r in [("L0 = 0.1", pulse()), ("L0 = 0.2", pulse("L0", 2))]:
    print(f"{label}: time of peak = {r[0]:.1f} min, peak height = {r[1]:.3f} uM, adapted level = {r[2]:.3f} uM")
model.set_param("L0", 10)     # restore the default